# 第2章：数据系统基础

## 本章学习目标

- 掌握 qlib 数据架构
- 熟练使用核心数据 API
- 理解数据缓存机制
- 学会使用表达式引擎

---

## 2.1 数据架构概览

Qlib 数据系统采用三层架构：

```
┌─────────────────────────────────────────────────────┐
│                    用户 API 层                       │
│            D.features(), D.calendar()               │
├─────────────────────────────────────────────────────┤
│                   Provider 层                        │
│     LocalProvider, FileStorage, PITProvider         │
├─────────────────────────────────────────────────────┤
│                   Storage 层                         │
│       文件存储、数据库存储、二进制格式               │
└─────────────────────────────────────────────────────┘
```

- **API 层**：提供统一的数据访问接口
- **Provider 层**：负责数据读取和缓存管理
- **Storage 层**：数据的持久化存储

In [ ]:
import qlib
from qlib.data import D
import pandas as pd
import numpy as np

# 初始化 qlib
qlib.init(
    provider_uri="~/.qlib/qlib_data/cn_data",
    region="cn",
)

print("Qlib 初始化成功")

## 2.2 核心数据 API

### 2.2.1 交易日历 `D.calendar()`

In [ ]:
# 获取日频率交易日历
calendar_day = D.calendar(freq="day")
print(f"日频交易日历: {len(calendar_day)} 个交易日")
print(f"时间范围: {calendar_day[0]} ~ {calendar_day[-1]}")

In [ ]:
# 指定时间范围
calendar_range = D.calendar(
    freq="day",
    start_time="2023-01-01",
    end_time="2023-12-31"
)
print(f"2023年交易日数量: {len(calendar_range)}")
print(f"\n前 5 个交易日: {calendar_range[:5]}")
print(f"后 5 个交易日: {calendar_range[-5:]}")

In [ ]:
# 不同频率的交易日历
calendar_week = D.calendar(freq="week")
print(f"周频交易日历: {len(calendar_week)} 周")

calendar_month = D.calendar(freq="month")
print(f"月频交易日历: {len(calendar_month)} 月")

### 2.2.2 股票列表 `D.instruments()`

In [ ]:
# 获取沪深300成分股
csi300 = D.instruments(market="csi300")
print(f"沪深300成分股数量: {len(csi300)}")
csi300.head(10)

In [ ]:
# 获取全市场股票
all_stocks = D.instruments(market="all")
print(f"全市场股票数量: {len(all_stocks)}")

# 获取指定时间点的成分股
csi300_2023 = D.instruments(
    market="csi300",
    start_time="2023-06-01",
    end_time="2023-06-01"
)
print(f"2023年6月沪深300成分股: {len(csi300_2023)}")

### 2.2.3 特征数据 `D.features()`

In [ ]:
# 基本用法：获取单只股票数据
df_single = D.features(
    instruments="SH600000",
    fields=["$close", "$open", "$high", "$low", "$volume", "$factor"],
    start_time="2023-01-01",
    end_time="2023-12-31",
    freq="day"
)

print(f"数据形状: {df_single.shape}")
df_single.head()

In [ ]:
# 获取多只股票数据
df_multi = D.features(
    instruments=["SH600000", "SH600016", "SH600019"],
    fields=["$close", "$volume"],
    start_time="2023-01-01",
    end_time="2023-03-31",
    freq="day"
)

print(f"数据形状: {df_multi.shape}")
print(f"\n索引层级: {df_multi.index.names}")
df_multi.head(10)

In [ ]:
# 获取市场成分股数据
df_market = D.features(
    instruments="csi300",  # 使用市场名称
    fields=["$close", "$volume"],
    start_time="2023-12-01",
    end_time="2023-12-31",
    freq="day"
)

print(f"数据形状: {df_market.shape}")
print(f"股票数量: {len(df_market.index.get_level_values('instrument').unique())}")
df_market.head()

## 2.3 数据字段说明

Qlib 内置数据集包含以下主要字段：

| 字段 | 说明 |
|------|------|
| `$open` | 开盘价 |
| `$close` | 收盘价 |
| `$high` | 最高价 |
| `$low` | 最低价 |
| `$volume` | 成交量（股） |
| `$factor` | 复权因子 |
| `$vwap` | 成交量加权平均价 |
| `$amount` | 成交额（元） |
| `$turnover` | 换手率 |
| `$adjclose` | 后复权收盘价 |

In [ ]:
# 查看可用字段
from qlib.data import D

# 通过获取一条数据来查看字段
sample = D.features(
    instruments="SH600000",
    fields=["$open", "$close", "$high", "$low", "$volume", "$factor", "$vwap", "$amount"],
    start_time="2023-12-01",
    end_time="2023-12-31",
)

sample.head()

## 2.4 表达式引擎

Qlib 支持使用表达式来计算衍生特征，无需手动预计算。

### 2.4.1 基本表达式

In [ ]:
# 使用表达式计算收益率
df_expr = D.features(
    instruments="SH600000",
    fields=[
        "$close",
        "Ref($close, 1)",           # 前一日收盘价
        "($close / Ref($close, 1) - 1)",  # 日收益率
    ],
    start_time="2023-01-01",
    end_time="2023-12-31",
)

# 重命名列
df_expr.columns = ["close", "prev_close", "daily_return"]
df_expr.head()

In [ ]:
# 验证计算结果
df_expr["manual_return"] = df_expr["close"].pct_change()
df_expr[["daily_return", "manual_return"]].head()

### 2.4.2 常用操作符

In [ ]:
# 常用操作符示例
df_ops = D.features(
    instruments="SH600000",
    fields=[
        "$close",
        "Mean($close, 5)",         # 5日均线
        "Mean($close, 20)",        # 20日均线
        "Std($close, 20)",         # 20日标准差
        "Max($high, 20)",          # 20日最高价
        "Min($low, 20)",           # 20日最低价
        "Sum($volume, 5)",         # 5日成交量之和
        "Ref($close, 5)",          # 5日前收盘价
    ],
    start_time="2023-01-01",
    end_time="2023-12-31",
)

# 重命名列
df_ops.columns = ["close", "ma5", "ma20", "std20", "high20", "low20", "vol5", "close_5d_ago"]
df_ops.head(25)

### 2.4.3 技术指标表达式

In [ ]:
# 技术指标计算
df_tech = D.features(
    instruments="SH600000",
    fields=[
        "$close",
        # 布林带
        "Mean($close, 20)",                              # 中轨
        "Mean($close, 20) + 2 * Std($close, 20)",        # 上轨
        "Mean($close, 20) - 2 * Std($close, 20)",        # 下轨
        # 动量
        "$close / Ref($close, 20) - 1",                  # 20日动量
        # 波动率
        "Std($close / Ref($close, 1) - 1, 20)",          # 20日波动率
    ],
    start_time="2023-01-01",
    end_time="2023-12-31",
)

df_tech.columns = ["close", "boll_mid", "boll_upper", "boll_lower", "momentum_20d", "volatility_20d"]
df_tech.head(25)

## 2.5 数据缓存机制

Qlib 提供多层缓存来加速数据访问。

```
┌─────────────────────────────────────────┐
│           内存缓存 (最快)                │
│        CacheTemp, MemCache              │
├─────────────────────────────────────────┤
│           磁盘缓存 (较快)                │
│        FileStorage Cache                │
├─────────────────────────────────────────┤
│           原始数据 (最慢)                │
│        Binary/CSV Files                 │
└─────────────────────────────────────────┘
```

In [ ]:
import time

# 测试缓存效果
def test_cache_performance():
    """测试缓存对数据访问速度的影响"""
    
    # 第一次访问（可能需要从磁盘加载）
    start = time.time()
    df1 = D.features(
        instruments="csi300",
        fields=["$close", "$volume"],
        start_time="2020-01-01",
        end_time="2022-12-31",
    )
    time1 = time.time() - start
    
    # 第二次访问（应该命中缓存）
    start = time.time()
    df2 = D.features(
        instruments="csi300",
        fields=["$close", "$volume"],
        start_time="2020-01-01",
        end_time="2022-12-31",
    )
    time2 = time.time() - start
    
    print(f"第一次访问: {time1:.4f} 秒")
    print(f"第二次访问: {time2:.4f} 秒 (缓存)")
    print(f"加速比: {time1/time2:.2f}x")
    
test_cache_performance()

## 2.6 数据可视化实战

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

# 设置中文字体
plt.rcParams["font.sans-serif"] = ["Arial Unicode MS", "SimHei"]
plt.rcParams["axes.unicode_minus"] = False

# 获取数据
df = D.features(
    instruments="SH600000",
    fields=["$open", "$close", "$high", "$low", "$volume"],
    start_time="2023-01-01",
    end_time="2023-12-31",
)

# 计算均线
df["ma5"] = df["$close"].rolling(5).mean()
df["ma20"] = df["$close"].rolling(20).mean()

In [ ]:
# 绘制 K 线图和均线
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 10), gridspec_kw={'height_ratios': [3, 1]})

dates = df.index.get_level_values('datetime')

# 收盘价和均线
ax1.plot(dates, df['$close'], label='收盘价', color='black', linewidth=1)
ax1.plot(dates, df['ma5'], label='MA5', color='red', linewidth=1, alpha=0.7)
ax1.plot(dates, df['ma20'], label='MA20', color='blue', linewidth=1, alpha=0.7)

# 填充均线之间的区域
ax1.fill_between(dates, df['ma5'], df['ma20'], 
                  where=(df['ma5'] >= df['ma20']), 
                  color='red', alpha=0.1, label='多头区域')
ax1.fill_between(dates, df['ma5'], df['ma20'], 
                  where=(df['ma5'] < df['ma20']), 
                  color='green', alpha=0.1, label='空头区域')

ax1.set_title('SH600000 浦发银行 - 价格走势与均线 (2023)', fontsize=14)
ax1.set_xlabel('日期')
ax1.set_ylabel('价格 (元)')
ax1.legend(loc='upper left')
ax1.grid(True, alpha=0.3)

# 成交量
colors = ['red' if df['$close'].iloc[i] >= df['$open'].iloc[i] else 'green' 
          for i in range(len(df))]
ax2.bar(dates, df['$volume'], color=colors, alpha=0.6)
ax2.set_title('成交量', fontsize=12)
ax2.set_xlabel('日期')
ax2.set_ylabel('成交量')
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 2.7 实践练习

### 练习目标

1. 获取指定时间范围的交易日历
2. 查询沪深300成分股的行情数据
3. 使用表达式计算技术指标（MA5、MA20、RSI）
4. 绘制 K 线图和技术指标

In [ ]:
# 练习1: 计算 RSI 指标
# RSI = 100 - 100 / (1 + RS)
# RS = 平均上涨幅度 / 平均下跌幅度

# 提示：使用表达式计算，或者获取数据后手动计算

# 你的代码



# 参考答案（手动计算方式）
# df = D.features(instruments='SH600000', fields=['$close'], start_time='2023-01-01', end_time='2023-12-31')
# delta = df['$close'].diff()
# gain = delta.where(delta > 0, 0).rolling(14).mean()
# loss = (-delta.where(delta < 0, 0)).rolling(14).mean()
# rs = gain / loss
# df['rsi'] = 100 - (100 / (1 + rs))

In [ ]:
# 练习2: 获取沪深300所有成分股 2023年的收盘价
# 计算每只股票的年化收益率和波动率
# 找出收益率最高的 10 只股票

# 你的代码



# 参考答案
# df = D.features(instruments='csi300', fields=['$close'], start_time='2023-01-01', end_time='2023-12-31')
# # 按 instrument 分组计算
# returns = df['$close'].groupby('instrument').apply(lambda x: x.iloc[-1] / x.iloc[0] - 1)
# print(returns.nlargest(10))

In [ ]:
# 练习3: 绘制 SH600000 的 MACD 指标
# MACD = EMA(12) - EMA(26)
# Signal = EMA(MACD, 9)
# Histogram = MACD - Signal

# 你的代码



# 参考答案
# df = D.features(instruments='SH600000', fields=['$close'], start_time='2022-01-01', end_time='2023-12-31')
# ema12 = df['$close'].ewm(span=12, adjust=False).mean()
# ema26 = df['$close'].ewm(span=26, adjust=False).mean()
# macd = ema12 - ema26
# signal = macd.ewm(span=9, adjust=False).mean()
# histogram = macd - signal
# 
# fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 8))
# ax1.plot(df.index.get_level_values('datetime'), df['$close'], label='收盘价')
# ax2.plot(df.index.get_level_values('datetime'), macd, label='MACD')
# ax2.plot(df.index.get_level_values('datetime'), signal, label='Signal')
# ax2.bar(df.index.get_level_values('datetime'), histogram, label='Histogram', alpha=0.5)
# plt.legend()
# plt.show()

## 2.8 本章小结

本章我们学习了：

1. **数据架构**：Provider、Storage、Cache 三层结构
2. **核心 API**：
   - `D.calendar()` - 交易日历
   - `D.instruments()` - 股票列表
   - `D.features()` - 特征数据
3. **表达式引擎**：使用表达式动态计算衍生特征
4. **数据缓存**：多层次缓存机制加速数据访问

### 常用操作符速查

| 操作符 | 说明 | 示例 |
|--------|------|------|
| `Ref` | 引用历史值 | `Ref($close, 5)` |
| `Mean` | 移动平均 | `Mean($close, 20)` |
| `Std` | 标准差 | `Std($close, 20)` |
| `Sum` | 求和 | `Sum($volume, 5)` |
| `Max` | 最大值 | `Max($high, 20)` |
| `Min` | 最小值 | `Min($low, 20)` |
| `Corr` | 相关系数 | `Corr($close, $volume, 20)` |
| `Cov` | 协方差 | `Cov($close, $volume, 20)` |

### 下一章预告

下一章我们将学习 qlib 的工作流系统，包括：
- YAML 配置驱动的工作流
- `qrun` 命令行工具
- Recorder 实验管理模式